In [25]:
#import
import ConnectionConfig as cc
debugging_mode=True

In [26]:
#config
cc.setupEnvironment()
spark = cc.startLocalCluster("AnalyseVragenS2",4)
spark.getActiveSession()

Environment variables are set...


In [27]:
#EXTRACT

#Dimensies en fact inladen
dateDimDf= spark.read.format("delta").load("./delta/DATE_DIM")
rainDimDf= spark.read.format("delta").load("./delta/RAIN_DIM")
seasonDimDf= spark.read.format("delta").load("./delta/SEASON_DIM")
userDimDf = spark.read.format("delta").load("./spark-warehouse/dimuser")
treasureTypeDimDf = spark.read.format("delta").load("./spark-warehouse/dimtreasuretype")
treasureFoundFactDf = spark.read.format("delta").load("./delta/FACT_TREASURE_FOUND")

dateDimDf.createOrReplaceTempView("dimDate")
rainDimDf.createOrReplaceTempView("dimRain")
seasonDimDf.createOrReplaceTempView("dimSeason")
userDimDf.createOrReplaceTempView("dimUser")
treasureTypeDimDf.createOrReplaceTempView("dimTreasureType")
treasureFoundFactDf.createOrReplaceTempView("factTreasureFound")

In [28]:
#  Wat is de invloed van het type user op de duur van de treasurehunt?
spark.sql("""
    SELECT
        dU.experiencelevel,
        COUNT(*) as hunts,
        ROUND(AVG(fTF.duration) / 60, 2) as avg_duration_minutes
    FROM factTreasureFound fTF
    JOIN dimUser dU ON dU.userSurKey = fTF.UserSurKey
    WHERE dU.experiencelevel IS NOT NULL
    GROUP BY dU.experiencelevel
    ORDER BY dU.experiencelevel
""").show()

+---------------+---------------+--------------------+
|experiencelevel|number_of_hunts|avg_duration_minutes|
+---------------+---------------+--------------------+
|        Amateur|        1044541|               70.03|
|         Pirate|         654656|               70.03|
|   Professional|          90241|               70.11|
+---------------+---------------+--------------------+



In [29]:
#  Vinden users de cache gemiddeld sneller in de regen?
spark.sql("""
          SELECT dR.RainCode,
                 COUNT(*)                         as hunts,
                 ROUND(AVG(fTF.duration) / 60, 1) as avg_minutes
          FROM factTreasureFound fTF
                   JOIN dimRain dR ON dR.RainSurKey = fTF.RainSurKey
          WHERE dR.RainCode IN ('RAIN', 'NORAIN')
          GROUP BY dR.RainCode
          ORDER BY dR.RainCode DESC
          """).show()

+---------+-----+-----------+
|condition|hunts|avg_minutes|
+---------+-----+-----------+
+---------+-----+-----------+



In [30]:
#  Zoeken beginnende users gemiddeld naar grotere caches (aantal stages)?
spark.sql("""
          SELECT COUNT(*) as hunts,
                 dTT.size
          FROM factTreasureFound fTF
                   JOIN dimUser dU ON dU.userSurKey = fTF.UserSurKey
                   JOIN dimTreasureType dTT ON dTT.TreasureTypeSurKey = fTF.TreasureTypeSurKey
          WHERE dU.experiencelevel = 'Amateur'
          GROUP BY dTT.size
          order by hunts DESC
          """).show()

+------+----+
| hunts|size|
+------+----+
|391070|   1|
|141908|   5|
|103241|   4|
|102579|   6|
|101685|   7|
| 61019|   8|
| 41384|   9|
| 41331|   3|
| 40659|   2|
| 19665|  10|
+------+----+



In [31]:
# Users van welke landen proberen het meest de moeilijkste caches
spark.sql("""
          SELECT COUNT(*) as hunts,
                 dU.country
          FROM factTreasureFound fTF
                   JOIN dimUser dU ON dU.userSurKey = fTF.UserSurKey
                   JOIN dimTreasureType dTT ON dTT.TreasureTypeSurKey = fTF.TreasureTypeSurKey
          WHERE dTT.difficulty = 4
          GROUP BY dU.country
          order by hunts DESC
          """).show()


+-----+-------+
|hunts|country|
+-----+-------+
|83229|     IN|
|20977|     US|
|13104|     BR|
|11686|     PK|
|10619|     BD|
| 9525|     RU|
| 8552|     JP|
| 7898|     MX|
| 6297|     PH|
| 5456|     DE|
| 4900|     TR|
| 4455|     TH|
| 4424|     FR|
| 3972|     GB|
| 3845|     IT|
| 3470|     ZA|
| 3218|     CO|
| 3144|     UA|
| 3116|     ES|
| 2657|     AR|
+-----+-------+
only showing top 20 rows


In [32]:
# hoeveel van de "easy" caches worden gevonden door welke experiencelevel en per seizoen
spark.sql("""
          SELECT COUNT(*) as hunts,
                 dU.experiencelevel,
                 dS.SeasonName
          FROM factTreasureFound fTF
                   JOIN dimUser dU ON dU.userSurKey = fTF.UserSurKey
                   JOIN dimTreasureType dTT ON dTT.TreasureTypeSurKey = fTF.TreasureTypeSurKey
                   JOIN dimSeason dS ON dS.SeasonSurKey = fTF.SeasonSurKey
          WHERE dTT.difficulty in (0, 1)
          GROUP BY dU.experiencelevel,dS.SeasonName
          order by hunts DESC
          """).show()

+-----+---------------+----------+
|hunts|experiencelevel|SeasonName|
+-----+---------------+----------+
|71054|        Amateur|     Zomer|
|70978|        Amateur|     Lente|
|70566|        Amateur|    Herfst|
|69989|        Amateur|    Winter|
|44465|         Pirate|    Herfst|
|44370|         Pirate|     Zomer|
|44009|         Pirate|     Lente|
|43819|         Pirate|    Winter|
| 6132|   Professional|     Lente|
| 6124|   Professional|     Zomer|
| 6077|   Professional|    Herfst|
| 6031|   Professional|    Winter|
+-----+---------------+----------+

